# 01 — Build the team-season dataset

**Runs only on a `GO` or `GO-TIER-B` verdict** in `futures/artifacts/data_audit.json` (PREREGISTRATION §5).

Builds the modelling panel: one row per team-season, the target from §2.1, the market line from
§2.2, and only feature families the audit classified `AVAILABLE` in §2.3. Every column is stamped
with the season whose information it may see; a column that can see season *S* is a leak and fails
the build.

**Writes:** `futures/data/team_season_panel.parquet` + `futures/artifacts/dataset_metadata.json`
(feature list *in order*, per-column availability season, input hashes, as-of date, seed, row
counts by season).


> **STATUS: SCAFFOLD — no implementation.** The sections below are the planned structure, frozen for review before any code is written. Each will follow the repo's markdown → code → inline-test convention.

```bash
papermill futures/season_team_totals/01_build_dataset.ipynb /tmp/out.ipynb
```

## Parameters

In [ ]:
AUDIT_PATH   = None    # None -> futures/artifacts/data_audit.json
SEASON_MIN   = None    # None -> inherit the audited window
TARGET_SEASON = None   # None -> inherit the audit's predict season
WRITE_ARTIFACTS = True
SEED         = 20260802
RUN_TESTS    = True

## Planned sections

**Section 1 — Gate on the audit** — Load `data_audit.json`; abort on `NO-GO`. Read the frozen fold set, the complete-season list, the predict season, and the `AVAILABLE` feature families. Nothing in this notebook may re-decide any of them — a fold set chosen here would be chosen after seeing data.

**Section 2 — Outcome join** — Rebuild the team-season outcome table from the same snapshot the audit hashed, and assert the hash matches. `wins_half_ties` is the target; strict `wins`, `ties`, and `games_played` ride along for re-grading and for denominators.

**Section 3 — Market line join** — Join the validated preseason lines on (season, franchise). Keep `win_total_line`, both prices, `book`, `as_of_date`. Rows without a line are retained but flagged `line_covered=False`: they may train, they may never score a market comparison.

**Section 4 — Prior-season features (AVAILABLE families only)** — Record, point differential, Pythagorean expectation, off/def EPA per play, turnover and third-down rates — all from season S−1 and earlier, all shifted, none touching season S. Multi-season decayed form from S−3..S−1.

**Section 5 — Schedule-structure features** — From season S's *published* schedule only: opponent identity, home/away split, rest, division/conference structure, bye placement. Opponent STRENGTH is measured from S−1 results — never from S.

**Section 6 — Coach features** — Week-1 coach of record; career and rolling win% computed through S−1 only.

**Section 7 — Leakage tests (the section that matters)** — Per column: (a) declared availability season ≤ S−1, or the column is schedule-structure; (b) a shuffled-outcome control — refit-free correlation of each feature against the season-S target must not exceed its correlation under a season-permuted target beyond chance; (c) an explicit assertion that no season-S result, score, or PBP column survives into the feature matrix by name or by construction; (d) the predict season is present with features and a null target.

**Section 8 — Freeze and write** — Pin `FEATURE_COLS` order (order is contract in this repo — reordering changes model identity), write the panel + metadata with input hashes, as-of date, and seed.

## Not implemented

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SCAFFOLD — not implemented yet. Only `00_data_audit.ipynb` carries code today
# (Joseph reviews the audit before anything downstream is built), and the audit's
# current verdict is the gate this notebook would have to clear first.
#
# When implemented, this cell becomes the §5 gate: read futures/artifacts/data_audit.json,
# refuse to run on a NO-GO verdict, and read the FROZEN fold sets from it (headline + the A1.4 strict-subset sensitivity, which every reported number must carry) rather than
# choosing folds here.
# ─────────────────────────────────────────────────────────────────────────────
raise NotImplementedError(
    "futures/season_team_totals/01_build_dataset.ipynb is a scaffold. Sections are planned in the markdown cells above; "
    "implementation is gated on (1) review of 00_data_audit.ipynb and (2) a GO verdict "
    "in futures/artifacts/data_audit.json."
)

## Conclusion and next steps

**Status: scaffold — nothing implemented, nothing decided.** This notebook has produced no result and
written no artifact.

**Gate in force:** `futures/artifacts/data_audit.json` reads **`GO-TIER-B`** (2026-08-03) under
`PREREGISTRATION.md` §10 Amendment 1 — §7 gates **A and B** only, `tier_c_open: false`. The frozen
fold sets are the headline (10 test seasons) and the mandatory A1.4 strict-subset sensitivity
(4 test seasons, underpowered); both are read from the artifact, never recomputed here.

**On implementation this notebook must:** follow the repo's markdown → code → inline-test structure
with an explanation above and an interpretation below **every** code cell; report every headline
number twice (headline and A1.4 sensitivity); name the benchmark an *archived market consensus of
unattributed sportsbook origin*; and carry the §7 language fence — no sides, probabilities against a
posted line, confidence tiers, EV, or profitability, and none of the words *bet*, *edge*, *lock*,
*value*, *play*.

**Next step:** implement the sections planned above, in order, after the preceding notebook in the
pipeline has run and its artifact exists.